# 전처리

In [2]:
# 데이터 불러오기
import os
import json
from glob import glob
import pandas as pd

data_dir = "../data/"
train_data_path = os.path.join(data_dir, "train.csv")
test_data_path = os.path.join(data_dir, "test.csv")
dev_data_path = os.path.join(data_dir, "dev.csv")

train_data = pd.read_csv(train_data_path)
test_data = pd.read_csv(test_data_path)
dev_data = pd.read_csv(dev_data_path)

print(f"✅ 학습 데이터 개수: {len(train_data)}")
print(f"✅ 테스트 데이터 개수: {len(test_data)}")
print(f"✅ 검증 데이터 개수: {len(dev_data)}")

✅ 학습 데이터 개수: 12457
✅ 테스트 데이터 개수: 499
✅ 검증 데이터 개수: 499


In [3]:
import re

def preprocess(text, is_input=True):
    # 1. HTML 태그 처리
    text = text.replace('<br>', '\n')
    text = text.replace('\\n', '\n')
    text = re.sub(r'<[^>]+>', '', text)  # 기타 태그 제거

    # # 2. 특수 문자 제거 (대화 구조를 해치지 않는 선에서 최소화)
    # # 보존할 문자: #, :, ?, !, ., ,, (), -, ', /
    # text = re.sub(r"[^a-zA-Z0-9가-힣\s\.,\?!#:\(\)\-'/]", '', text)

    # 3. 줄바꿈/공백 정리
    text = re.sub(r'\n+', '\n', text)  # 여러 줄바꿈 → 1개
    text = re.sub(r'[ ]{2,}', ' ', text)  # 다중 스페이스 제거

    # 4. 스피커 태그 뒤에는 항상 한 칸 띄우기 (정규화)
    text = re.sub(r'(#Person\d+#:)\s*', r'\1 ', text)

    # 5. 맨 앞에 "summarize: " prefix 추가
    if is_input:
        text = "summarize: " + text

    return text.strip()



In [5]:
# 1. Train 데이터 (입력 & 정답 둘 다!)
# ------------------------------------------------
# 입력(Dialogue): 노이즈 제거 + Prefix("요약: ") 추가
train_data['dialogue'] = train_data['dialogue'].apply(lambda x: preprocess(x, is_input=True))
# 정답(Summary): 노이즈 제거만 (Prefix 없음!)
train_data['summary'] = train_data['summary'].apply(lambda x: preprocess(x, is_input=False))

# 2. Dev (Validation) 데이터 (입력 & 정답 둘 다!)
# ------------------------------------------------
dev_data['dialogue'] = dev_data['dialogue'].apply(lambda x: preprocess(x, is_input=True))
dev_data['summary'] = dev_data['summary'].apply(lambda x: preprocess(x, is_input=False))

# 3. Test 데이터 (입력만!)
# ------------------------------------------------
# 시험지니까 입력만 깨끗하게 닦아서 줍니다. (정답은 우리가 만들어야 하니까요)
test_data['dialogue'] = test_data['dialogue'].apply(lambda x: preprocess(x, is_input=True))

In [6]:
# 새로운 전처리 데이터 저장
train_data.to_csv(os.path.join(data_dir, 'processed_prefix', "train_preprocessed.csv"), index=False)
dev_data.to_csv(os.path.join(data_dir, 'processed_prefix', "dev_preprocessed.csv"), index=False)
test_data.to_csv(os.path.join(data_dir, 'processed_prefix', "test_preprocessed.csv"), index=False)  

In [ ]:
# 증강된 데이터 + 원본 데이터 합친 후 전처리
train_aug_full_path = os.path.join(data_dir, "augmented", "train_augmented_full.csv")
train_aug_full = pd.read_csv(train_aug_full_path)

# 입력(Dialogue): 노이즈 제거 + Prefix("요약: ") 추가
train_aug_full['dialogue'] = train_aug_full['dialogue'].apply(lambda x: preprocess(x, is_input=True))
# 정답(Summary): 노이즈 제거만 (Prefix 없음!)
train_aug_full['summary'] = train_aug_full['summary'].apply(lambda x: preprocess(x, is_input=False))

train_aug_full.to_csv(os.path.join(data_dir, "processed_prefix", "train_augmented_full.csv"), index=False)

In [4]:
len(train_aug_full)

24902

In [45]:
# dialogue에 한자 섞인 데이터 있는지 확인
def contains_hanja(text):
    for ch in text:
        code = ord(ch)
        # 기본 한자 + 확장 한자 영역 모두 포함
        if (
            0x4E00 <= code <= 0x9FFF or     # CJK Unified Ideographs
            0x3400 <= code <= 0x4DBF or     # CJK Unified Ideographs Extension A
            0x20000 <= code <= 0x2A6DF or   # Extension B
            0x2A700 <= code <= 0x2B73F or   # Extension C
            0x2B740 <= code <= 0x2B81F or   # Extension D
            0x2B820 <= code <= 0x2CEAF or   # Extension E
            0x2CEB0 <= code <= 0x2EBEF or   # Extension F
            0x30000 <= code <= 0x3134F      # Extension G
        ):
            return True
    return False

augmented_data['has_hanja'] = augmented_data['dialogue'].apply(contains_hanja)
# fname 출력
hanja_files = augmented_data[augmented_data['has_hanja'] == True]
print("한자 포함된 데이터 개수:", len(hanja_files))

h_path = "./hanja.csv"
hanja_files['fname'].to_csv(h_path, index=False, encoding="utf-8-sig")

# dialogue에 일어 섞인 데이터 있는지 확인
def contains_japanese(text):
    return bool(re.search(r'[\u3040-\u30FF\u31F0-\u31FF\uFF66-\uFF9D]', text))
augmented_data['has_japanese'] = augmented_data['dialogue'].apply(contains_japanese)
# fname 출력
japanese_files = augmented_data[augmented_data['has_japanese'] == True]
print("일본어 포함된 데이터 개수:", len(japanese_files))
print(japanese_files['fname'])

한자 포함된 데이터 개수: 3
일본어 포함된 데이터 개수: 0
Series([], Name: fname, dtype: object)
